In [1]:
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df = df.drop('customerID', axis=1)

X = df.drop('Churn', axis=1)
y = (df['Churn'] == 'Yes').astype(int)

categorical_cols = X.select_dtypes(include='object').columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)

numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
scaler.fit(X_train[numeric_cols])

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(X_train.columns.tolist(), '../models/feature_columns.pkl')

print("Saved scaler and feature columns")
print("Number of columns:", len(X_train.columns))

C:\Users\chint\AppData\Local\Temp\ipykernel_1256\1516437855.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include='object').columns.tolist()


Saved scaler and feature columns
Number of columns: 30


In [2]:
# Add the same 3 engineered features from Phase 4, using original unscaled values
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

X_train_fe['AvgMonthlySpend'] = df.loc[y_train.index, 'TotalCharges'] / (df.loc[y_train.index, 'tenure'] + 1)
X_test_fe['AvgMonthlySpend'] = df.loc[y_test.index, 'TotalCharges'] / (df.loc[y_test.index, 'tenure'] + 1)

bins = [-1, 12, 24, 48, 100]
labels = ['0-12mo', '13-24mo', '25-48mo', '49mo+']
tenure_group_train = pd.cut(df.loc[y_train.index, 'tenure'], bins=bins, labels=labels)
tenure_group_test = pd.cut(df.loc[y_test.index, 'tenure'], bins=bins, labels=labels)
tenure_dummies_train = pd.get_dummies(tenure_group_train, prefix='TenureGroup', drop_first=True)
tenure_dummies_test = pd.get_dummies(tenure_group_test, prefix='TenureGroup', drop_first=True)
X_train_fe = pd.concat([X_train_fe.reset_index(drop=True), tenure_dummies_train.reset_index(drop=True)], axis=1)
X_test_fe = pd.concat([X_test_fe.reset_index(drop=True), tenure_dummies_test.reset_index(drop=True)], axis=1)

service_cols = ['OnlineSecurity_Yes', 'OnlineBackup_Yes', 'DeviceProtection_Yes', 'TechSupport_Yes', 'StreamingTV_Yes', 'StreamingMovies_Yes']
X_train_fe['NumServices'] = X_train_fe[service_cols].sum(axis=1)
X_test_fe['NumServices'] = X_test_fe[service_cols].sum(axis=1)

# Re-save the CORRECT, final feature column list (35 columns, matching the deployed model)
joblib.dump(X_train_fe.columns.tolist(), '../models/feature_columns.pkl')

print("Corrected number of columns:", len(X_train_fe.columns))
print(X_train_fe.columns.tolist())

Corrected number of columns: 35
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'AvgMonthlySpend', 'TenureGroup_13-24mo', 'TenureGroup_25-48mo', 'TenureGroup_49mo+', 'NumServices']
